# TranAD 在 WADI 上的复现实验

本笔记本与仓库根目录下 `TranAD/` 中的 `main.py`、`preprocess.py`、`src/models.py` 等逻辑一致，用于在 **WADI** 数据上训练与评估 **TranAD** 模型。

**环境与依赖（`TranAD/requirements.txt`）**

1. **先确认当前 Jupyter 用的 Python 里能 `import torch`**。若没有 PyTorch，在终端（建议先 `conda activate` 你跑笔记本的环境）执行例如：
   ```text
   pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
   ```
   （需要 GPU 版请到 [pytorch.org](https://pytorch.org) 选对应命令。）

2. **再安装 TranAD 其余库**（与仓库一致）：
   ```text
   cd TranAD
   pip install -r requirements.txt
   ```
   其中包含：`pandas`、`tqdm`、`matplotlib`、`dgl`、`numpy`、`SciencePlots`、`scikit-learn`、`xlrd` 等；`requirements.txt` 内也有简短注释。

3. **内核**：选「已装好上述包」的那个 Python / conda 环境作为 Notebook kernel 即可。

**数据**

- 将官方 WADI 放到 `TranAD/data/WADI/`，或在下文配置 **自定义 CSV**（`Date`/`Time` + 第 4 列起为传感器；你也可用「测试集带点级标签列」的 A2 流程）。
- 预处理生成 `TranAD/processed/WADI/{train,test,labels}.npy`；训练时通过 `os.chdir(TranAD)` 读写 `processed/` 与 `checkpoints/`。

环境与运行说明见项目根目录 README.md。

## 1. 配置：TranAD 根目录与命令行参数

`--retrain` 表示重新训练；若已有 `checkpoints/TranAD_WADI/model.ckpt` 且只想测试，可改为 `--test`（并去掉 `--retrain`）。

In [ ]:
from pathlib import Path
import os
import os
import sys
from pathlib import Path

# 修改为本地 TranAD 仓库绝对路径
TRANAD_ROOT = Path(os.environ.get("TRANAD_ROOT", "../TranAD")).resolve()

assert TRANAD_ROOT.is_dir(), f"未找到 TranAD 目录: {TRANAD_ROOT}"

os.chdir(TRANAD_ROOT)
if str(TRANAD_ROOT) not in sys.path:
    sys.path.insert(0, str(TRANAD_ROOT))

# 须在首次 import TranAD 内模块之前设置（与命令行等价）
sys.argv = [
    "tranad_wadi_reproduce.ipynb",
    "--dataset", "WADI",
    "--model", "TranAD",
    "--retrain",
]

print("cwd:", os.getcwd())
print("argv:", sys.argv)

# =========================
# Matplotlib 中文字体配置（避免图中中文空白方框）
# =========================
import logging
import warnings
import matplotlib as mpl
from matplotlib import font_manager as fm

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="findfont: Generic family")
warnings.filterwarnings("ignore", message=r"Glyph .* missing from font\(s\)")

try:
    available = {f.name for f in fm.fontManager.ttflist}
except Exception:
    available = set()

preferred_cn = [
    "Microsoft YaHei",
    "SimHei",
    "SimSun",
    "Noto Sans CJK SC",
    "PingFang SC",
]
cn_fonts = [f for f in preferred_cn if f in available]
if not cn_fonts:
    cn_fonts = ["Microsoft YaHei", "SimHei", "SimSun"]

mpl.rcParams["font.family"] = "sans-serif"
# 中文字体优先，否则会回退到缺字的 DejaVu Sans
mpl.rcParams["font.sans-serif"] = cn_fonts + ["Arial", "DejaVu Sans"]
mpl.rcParams["axes.unicode_minus"] = False

print("Matplotlib 中文字体已配置（优先：{}）。".format(cn_fonts[0] if cn_fonts else "fallback"))

In [ ]:
# 重要：放在任何绘图/导入 src.plotting 之前，避免字体相关刷屏
import logging
import warnings
import matplotlib as mpl
from matplotlib import font_manager as fm

# 0) 关掉 font_manager 的噪声日志
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

# 1) 抑制两类常见字体警告（Times 找不到 / 中文 glyph 缺失）
warnings.filterwarnings("ignore", message="findfont: Generic family")
warnings.filterwarnings("ignore", message=r"Glyph .* missing from font\(s\)")

# 2) 自动选择本机可用中文字体（优先）
try:
    available = {f.name for f in fm.fontManager.ttflist}
except Exception:
    available = set()

preferred_cn = [
    "Microsoft YaHei",  # 微软雅黑
    "SimHei",           # 黑体
    "SimSun",           # 宋体
    "PingFang SC",
    "Noto Sans CJK SC",
]

cn_fonts = [f for f in preferred_cn if f in available]
if not cn_fonts:
    # 最差情况：仍然给一个列表，后续会回退
    cn_fonts = ["Microsoft YaHei", "SimHei", "SimSun"]

# 3) 统一设置字体回退链
mpl.rcParams["font.family"] = "sans-serif"
# 把中文字体放最前面，避免被 DejaVu Sans 抢先
mpl.rcParams["font.sans-serif"] = cn_fonts + ["Arial", "DejaVu Sans"]
# 给 serif 一个可用兜底，避免 generic serif 去找 Times
mpl.rcParams["font.serif"] = ["DejaVu Serif", "Times New Roman", "SimSun"]

mpl.rcParams["axes.unicode_minus"] = False

print("Matplotlib 字体已设置：", mpl.rcParams["font.sans-serif"])
print("已抑制 findfont(Times) 与 Glyph missing 警告。")

## 2.（可选）自定义 WADI 文件路径并写入 `processed/WADI/`

**情况 A — 官方三文件**（`WADI_14days.csv`、`WADI_attackdata.csv`、`WADI_attacklabels.csv`）：在下方把 `CUSTOM_*` 设为路径，或把文件放进 `TranAD/data/WADI/` 后运行 `python preprocess.py WADI`。

**情况 B — 仅训练 CSV + 带标签的测试 CSV**（例如 iTrust A2：`WADI_14days_new.csv` 与 `WADI_attackdataLABLE.csv`，最后一列为点级标签，**1=正常、-1=攻击**）：设置 `WADI_A2_DIR` 并令 `USE_WADI_POINT_LABEL_CSV = True`。脚本会跳过首行无效表头、对齐 `Row/Date/Time` 后第 4 列起的传感器，并按 TranAD 同样规则 **每 10 行取 1 行** 与 **按列 min-max 归一化**（与 `convertNumpy` 一致）。

In [ ]:
from pathlib import Path
import os
from pathlib import Path
from typing import List, Optional, Tuple, Union

import numpy as np
import pandas as pd

from src.folderconstants import output_folder


def convertNumpy(df: pd.DataFrame) -> np.ndarray:
    x = df[df.columns[3:]].values[::10, :]
    return (x - x.min(0)) / (x.ptp(0) + 1e-4)


def preprocess_wadi_custom(
    train_csv: Path,
    test_csv: Path,
    attack_labels_csv: Path,
    train_skiprows: int = 1000,
    train_nrows: Union[int, float] = 2e5,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """与 preprocess.py 中 WADI 分支一致，输出 train/test/labels 数组。"""
    ls = pd.read_csv(attack_labels_csv)
    train = pd.read_csv(train_csv, skiprows=train_skiprows, nrows=int(train_nrows))
    test = pd.read_csv(test_csv)
    train.dropna(how="all", inplace=True)
    test.dropna(how="all", inplace=True)
    train.fillna(0, inplace=True)
    test.fillna(0, inplace=True)
    test["Time"] = test["Time"].astype(str)
    test["Time"] = pd.to_datetime(test["Date"] + " " + test["Time"])
    labels = test.copy(deep=True)
    for i in test.columns.tolist()[3:]:
        labels[i] = 0
    for i in ["Start Time", "End Time"]:
        ls[i] = ls[i].astype(str)
        ls[i] = pd.to_datetime(ls["Date"] + " " + ls[i])
    for index, row in ls.iterrows():
        to_match = row["Affected"].split(", ")
        matched = []
        for i in test.columns.tolist()[3:]:
            for tm in to_match:
                if tm in i:
                    matched.append(i)
                    break
        st, et = str(row["Start Time"]), str(row["End Time"])
        labels.loc[(labels["Time"] >= st) & (labels["Time"] <= et), matched] = 1
    train_a, test_a, lab_a = convertNumpy(train), convertNumpy(test), convertNumpy(labels)
    print("train, test, labels:", train_a.shape, test_a.shape, lab_a.shape)
    return train_a, test_a, lab_a


def _norm_downsample_10(arr: np.ndarray) -> np.ndarray:
    x = arr[::10, :].astype(np.float64)
    return (x - x.min(0)) / (x.ptp(0) + 1e-4)


def preprocess_wadi_point_label_csv(
    train_csv: Path,
    test_csv: Path,
    train_nrows: Optional[Union[int, float]] = 2e5,
    train_skiprows: int = 0,
    test_skiprows: Optional[List[int]] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """测试 CSV 含点级标签列（1=正常，-1=攻击）；自动跳过测试文件首行无效表头。"""
    if test_skiprows is None:
        test_skiprows = [0]
    nrows = int(train_nrows) if train_nrows is not None else None
    train = pd.read_csv(train_csv, skiprows=train_skiprows, nrows=nrows)
    test = pd.read_csv(test_csv, skiprows=test_skiprows)
    train.columns = train.columns.str.strip()
    test.columns = test.columns.str.strip()
    label_col = None
    for c in test.columns:
        low = str(c).lower()
        if "attack" in low and ("lable" in low or "label" in low):
            label_col = c
            break
    if label_col is None:
        label_col = test.columns[-1]
    feats = list(train.columns[3:])
    miss = [f for f in feats if f not in test.columns]
    if miss:
        raise ValueError("测试集缺少与训练集对齐的传感器列（示例）: " + repr(miss[:5]))
    train_x = train[feats].fillna(0).values.astype(np.float64)
    test_x = test[feats].fillna(0).values.astype(np.float64)
    train_a = _norm_downsample_10(train_x)
    test_a = _norm_downsample_10(test_x)
    raw_lab = test[label_col].values[::10]
    raw_lab = pd.to_numeric(pd.Series(raw_lab), errors="coerce").fillna(1.0).values
    y = (raw_lab < 0).astype(np.float64)
    labels = np.tile(y.reshape(-1, 1), (1, test_a.shape[1]))
    print("train, test, labels:", train_a.shape, test_a.shape, labels.shape, "| label列:", label_col)
    return train_a, test_a, labels


# ---------- A2：训练 CSV + 带标签测试 CSV（你的路径）----------
WADI_A2_DIR = Path(os.environ.get("WADI_DATA_ROOT", "../data/WADI"))
USE_WADI_POINT_LABEL_CSV = True
TRAIN_NROWS = 2e5
TRAIN_SKIPROWS = 0

# ---------- 官方三文件（与上互斥：改用官方时设 USE_WADI_POINT_LABEL_CSV = False）----------
CUSTOM_TRAIN = None
CUSTOM_TEST = None
CUSTOM_LABELS = None

folder = Path(output_folder) / "WADI"
folder.mkdir(parents=True, exist_ok=True)

if USE_WADI_POINT_LABEL_CSV:
    tr_path = WADI_A2_DIR / "WADI_14days_new.csv"
    te_path = WADI_A2_DIR / "WADI_attackdataLABLE.csv"
    if not tr_path.is_file() or not te_path.is_file():
        raise FileNotFoundError(f"请检查路径: {tr_path} , {te_path}")
    tr, te, lb = preprocess_wadi_point_label_csv(
        tr_path, te_path, train_nrows=TRAIN_NROWS, train_skiprows=TRAIN_SKIPROWS
    )
    np.save(folder / "train.npy", tr)
    np.save(folder / "test.npy", te)
    np.save(folder / "labels.npy", lb)
    print("已写入:", folder)
elif CUSTOM_TRAIN is not None and CUSTOM_TEST is not None and CUSTOM_LABELS is not None:
    tr, te, lb = preprocess_wadi_custom(CUSTOM_TRAIN, CUSTOM_TEST, CUSTOM_LABELS)
    np.save(folder / "train.npy", tr)
    np.save(folder / "test.npy", te)
    np.save(folder / "labels.npy", lb)
    print("已写入:", folder)
else:
    print("未启用 USE_WADI_POINT_LABEL_CSV 且未设置三文件 CUSTOM_*。请运行 preprocess.py 或打开 USE_WADI_POINT_LABEL_CSV。")

## 3. 若尚未生成 `processed/WADI`，在终端执行预处理

在 `TranAD` 目录下：
```bash
python preprocess.py WADI
```

下面单元格会检查 `processed/WADI` 是否存在。

In [ ]:
from pathlib import Path
import os
from pathlib import Path

wadi_proc = Path("processed") / "WADI"
need = [wadi_proc / "train.npy", wadi_proc / "test.npy", wadi_proc / "labels.npy"]
missing = [p for p in need if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "缺少预处理文件: " + ", ".join(str(p) for p in missing)
        + "\n请在 TranAD 目录运行: python preprocess.py WADI\n"
        "或完成上一节的自定义路径预处理。"
    )
print("预处理文件就绪:", need)

## 4. 导入模块（依赖上文的 `sys.argv`）

若修改了 `--dataset` / `--model`，请**重启内核**后从第一节重新运行，避免 `constants` 缓存旧参数。

**顺序**：先完成 **4.1 自检** 一格，再运行 **import main** 那一格。

### 4.1 环境自检（DGL → torchdata.datapipes）\n
\n
先运行 **下面代码格** 做自检；通过后再运行 **再下一格** `import main`。若终端已装 torchdata 仍报错，请确认内核为 **(tsad)**。

In [ ]:
# Install dependencies explicitly using docs/reproduction.md before running.
import subprocess
import torchdata.datapipes
import pydantic


### 4.15 PyTorch 2.x 兼容补丁（is_causal）

若运行第 5 节训练时报错 `forward() got an unexpected keyword argument 'is_causal'`，说明 PyTorch 2.x 与 TranAD 不兼容。**必须先运行下面代码格**，再执行第 5 节。

In [ ]:
# TranAD 使用 src.dlutils 中的自定义 Encoder/Decoder Layer，须先导入再打补丁
import src.dlutils as dlutils

# PyTorch 2.x 的 nn.TransformerEncoder/Decoder 会传入 is_causal，dlutils 中的 Layer 签名不支持
_enc_orig = dlutils.TransformerEncoderLayer.forward
def _enc_patched(self, src, src_mask=None, src_key_padding_mask=None, **kwargs):
    kwargs.pop("is_causal", None)
    return _enc_orig(self, src, src_mask, src_key_padding_mask)
dlutils.TransformerEncoderLayer.forward = _enc_patched

_dec_orig = dlutils.TransformerDecoderLayer.forward
def _dec_patched(self, tgt, memory, tgt_mask=None, memory_mask=None,
                 tgt_key_padding_mask=None, memory_key_padding_mask=None, **kwargs):
    kwargs.pop("is_causal", None)
    return _dec_orig(self, tgt, memory, tgt_mask, memory_mask,
                     tgt_key_padding_mask, memory_key_padding_mask)
dlutils.TransformerDecoderLayer.forward = _dec_patched

print("PyTorch is_causal 兼容补丁已应用（dlutils TransformerEncoder/DecoderLayer），可继续运行第 5 节。")

In [ ]:
import pickle
import importlib

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from time import time
from pprint import pprint
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, TensorDataset

# 确保 constants 使用当前 argv（首次运行通常无需 reload）
import src.constants as constants
importlib.reload(constants)

from src.constants import *
from src.pot import pot_eval
from src.diagnosis import hit_att, ndcg
from src.plotting import plotter
from src.utils import plot_accuracies

from main import (
    load_dataset,
    convert_to_windows,
    backprop,
    save_model,
    load_model,
)

constants.preds.clear()
print("dataset:", args.dataset, "model:", args.model, "retrain:", args.retrain, "test:", args.test)

## 5. 训练与测试（与 `main.py` 主流程一致）

- 训练 5 个 epoch（与仓库默认相同）。
- 评估使用 POT + `src/constants.py` 中 WADI 的 `lm` 与 `lr` 等超参。

In [ ]:
train_loader, test_loader, labels = load_dataset(args.dataset)

model, optimizer, scheduler, epoch, accuracy_list = load_model(args.model, labels.shape[1])

trainD, testD = next(iter(train_loader)), next(iter(test_loader))
trainO, testO = trainD, testD
trainD, testD = convert_to_windows(trainD, model), convert_to_windows(testD, model)

if not args.test:
    print(f"Training {args.model} on {args.dataset}")
    num_epochs = 5
    start = time()
    for e in tqdm(list(range(epoch + 1, epoch + num_epochs + 1)), desc="epochs"):
        lossT, lr = backprop(e, model, trainD, trainO, optimizer, scheduler)
        accuracy_list.append((lossT, lr))
    print("Training time: {:10.4f} s".format(time() - start))
    save_model(model, optimizer, scheduler, e, accuracy_list)
    plot_accuracies(accuracy_list, f"{args.model}_{args.dataset}")

model.eval()
print(f"Testing {args.model} on {args.dataset}")
with torch.no_grad():
    loss, y_pred = backprop(0, model, testD, testO, optimizer, scheduler, training=False)

if not args.test:
    if "TranAD" in model.name:
        testO_plot = torch.roll(testO, 1, 0)
    else:
        testO_plot = testO
    plotter(f"{args.model}_{args.dataset}", testO_plot, y_pred, loss, labels)

df = pd.DataFrame()
with torch.no_grad():
    lossT, _ = backprop(0, model, trainD, trainO, optimizer, scheduler, training=False)
for i in range(loss.shape[1]):
    lt, l, ls = lossT[:, i], loss[:, i], labels[:, i]
    result, pred = pot_eval(lt, l, ls)
    constants.preds.append(pred)
    df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)

lossTfinal, lossFinal = np.mean(lossT, axis=1), np.mean(loss, axis=1)
labelsFinal = (np.sum(labels, axis=1) >= 1) + 0
result, _ = pot_eval(lossTfinal, lossFinal, labelsFinal)
result.update(hit_att(loss, labels))
result.update(ndcg(loss, labels))
print(df)
pprint(result)

## 6. 内核说明与结果可视化

- **可以继续使用同一个 Jupyter kernel**（同一套 Python 解释器与环境），不必为画图单独换内核。
- 若你**修改了**第 1 节里的 `sys.argv`（例如 `--dataset` / `--model`），为避免 `constants` 仍用旧参数，建议 **Restart Kernel** 后从第 1 节顺序重跑。
- 下面画图依赖 **上一节「训练与测试」已成功执行**，内存中已有 `accuracy_list`、`lossFinal`、`labelsFinal`、`result` 等。若刚重启过内核，请先重跑预处理与训练节。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager as fm

# ===== 🔥 强制使用前面选好的中文字体 =====
if "cn_fonts" in globals() and len(cn_fonts) > 0:
    font_prop = fm.FontProperties(family=cn_fonts[0])
else:
    font_prop = None


# ---------- 1) 训练损失 ----------
try:
    al = accuracy_list
    if al is not None and len(al) > 0:
        fig, ax = plt.subplots(figsize=(8, 3.2))
        ax.plot([x[0] for x in al], marker="o", lw=1)
        ax.set_xlabel("epoch", fontproperties=font_prop)
        ax.set_ylabel("train L1", fontproperties=font_prop)
        ax.set_title("TranAD training loss", fontproperties=font_prop)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
except NameError:
    print("未找到 accuracy_list：请先运行「训练与测试」单元格。")


# ---------- 2) 测试集 ----------
try:
    scores = np.asarray(lossFinal, dtype=np.float64).ravel()
    y = np.asarray(labelsFinal, dtype=np.float64).ravel()
    t = np.arange(len(scores))

    fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    axes[0].plot(t, scores, lw=0.4, color="C0")
    axes[0].set_ylabel("mean score", fontproperties=font_prop)
    axes[0].set_title("Test: aggregate anomaly score (mean over dims)", fontproperties=font_prop)
    axes[0].grid(True, alpha=0.3)

    axes[1].fill_between(t, 0, 1, where=(y > 0.5), color="red", alpha=0.25, label="ground-truth anomaly")
    axes[1].set_ylim(-0.1, 1.1)
    axes[1].set_ylabel("label", fontproperties=font_prop)
    axes[1].set_xlabel("time index (after downsampling)", fontproperties=font_prop)
    axes[1].legend(loc="upper right", prop=font_prop)

    plt.tight_layout()
    plt.show()
except NameError as e:
    print("缺少 lossFinal / labelsFinal：请先运行训练单元格。", e)


# ---------- 3) 阈值 ----------
try:
    thr = float(result["threshold"])
    scores = np.asarray(lossFinal, dtype=np.float64).ravel()

    fig, ax = plt.subplots(figsize=(11, 3.2))
    ax.plot(scores, lw=0.35, color="C0", label="mean score")
    ax.axhline(thr, color="C1", ls="--", lw=1.2, label=f"POT threshold = {thr:.6f}")
    ax.set_xlabel("time index", fontproperties=font_prop)
    ax.set_ylabel("score", fontproperties=font_prop)
    ax.set_title("Aggregate score vs POT threshold", fontproperties=font_prop)
    ax.legend(loc="upper right", prop=font_prop)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
except (NameError, KeyError) as e:
    print("无法绘制阈值图：请先完成训练且 result 中含 threshold。", e)


# ---------- 4) 分布 ----------
try:
    s = np.asarray(lossFinal, dtype=np.float64).ravel()
    y = np.asarray(labelsFinal, dtype=np.float64).ravel()

    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.hist(s[y < 0.5], bins=60, alpha=0.55, density=True, label="normal")
    ax.hist(s[y > 0.5], bins=60, alpha=0.55, density=True, label="anomaly")
    ax.set_xlabel("score", fontproperties=font_prop)
    ax.set_ylabel("density", fontproperties=font_prop)
    ax.set_title("Score distribution", fontproperties=font_prop)
    ax.legend(prop=font_prop)

    plt.tight_layout()
    plt.show()
except NameError:
    print("跳过分布图：缺少 lossFinal / labelsFinal。")

### 6.1 混淆矩阵与扩展可视化

下方代码格会输出：**混淆矩阵热力图**、**POT 评估指标表格**、**ROC 曲线**、**PR 曲线**、**异常分数时序片段对比**、**TPR/FPR 随阈值变化** 等图表。需先运行第 5 节以生成 `lossFinal`、`labelsFinal`、`result`。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from matplotlib import font_manager as fm
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score
)

# ===== 🔥 强制中文字体（关键）=====
if "cn_fonts" in globals() and len(cn_fonts) > 0:
    matplotlib.rcParams["font.family"] = "sans-serif"
    matplotlib.rcParams["font.sans-serif"] = cn_fonts + ["Arial", "DejaVu Sans"]
    font_prop = fm.FontProperties(family=cn_fonts[0])
else:
    font_prop = None

matplotlib.rcParams["axes.unicode_minus"] = False

try:
    from src.pot import pot_eval
except ImportError:
    pot_eval = None

try:
    scores = np.asarray(lossFinal, dtype=np.float64).ravel()
    y_true = np.asarray(labelsFinal, dtype=np.float64).ravel()
    _, y_pred = pot_eval(lossTfinal, lossFinal, labelsFinal)

    fig_all = []

    # ----- 1) 混淆矩阵 -----
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)

    fig1, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].imshow(cm, cmap="Blues")
    axes[0].set_xticks([0, 1])
    axes[0].set_yticks([0, 1])
    axes[0].set_xticklabels(["正常", "异常"], fontproperties=font_prop)
    axes[0].set_yticklabels(["正常", "异常"], fontproperties=font_prop)
    axes[0].set_xlabel("预测", fontproperties=font_prop)
    axes[0].set_ylabel("真实", fontproperties=font_prop)
    axes[0].set_title("混淆矩阵（数量）", fontproperties=font_prop)

    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, str(cm[i, j]), ha="center", va="center", fontproperties=font_prop)

    axes[1].imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    axes[1].set_xticks([0, 1])
    axes[1].set_yticks([0, 1])
    axes[1].set_xticklabels(["正常", "异常"], fontproperties=font_prop)
    axes[1].set_yticklabels(["正常", "异常"], fontproperties=font_prop)
    axes[1].set_xlabel("预测", fontproperties=font_prop)
    axes[1].set_ylabel("真实", fontproperties=font_prop)
    axes[1].set_title("混淆矩阵（行归一化）", fontproperties=font_prop)

    for i in range(2):
        for j in range(2):
            axes[1].text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", fontproperties=font_prop)

    plt.tight_layout()
    plt.show()
    fig_all.append("混淆矩阵")

    # ----- 2) POT 指标 -----
    print("\n--- POT 评估指标 ---")
    for k, v in result.items():
        if isinstance(v, (int, float)) and not isinstance(v, bool):
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
    print(f"  TP={result.get('TP', '')} TN={result.get('TN', '')} FP={result.get('FP', '')} FN={result.get('FN', '')}\n")

    # ----- 3) ROC -----
    fpr, tpr, _ = roc_curve(y_true, scores)
    roc_auc_val = auc(fpr, tpr)

    fig2, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, lw=2, label=f"ROC (AUC = {roc_auc_val:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("假阳性率 (FPR)", fontproperties=font_prop)
    ax.set_ylabel("真阳性率 (TPR)", fontproperties=font_prop)
    ax.set_title("ROC 曲线", fontproperties=font_prop)
    ax.legend(prop=font_prop)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    fig_all.append("ROC")

    # ----- 4) PR -----
    prec, rec, _ = precision_recall_curve(y_true, scores)
    ap = average_precision_score(y_true, scores)

    fig3, ax = plt.subplots(figsize=(5, 4))
    ax.plot(rec, prec, lw=2, label=f"PR (AP = {ap:.3f})")
    ax.set_xlabel("召回率 (Recall)", fontproperties=font_prop)
    ax.set_ylabel("精确率 (Precision)", fontproperties=font_prop)
    ax.set_title("Precision-Recall 曲线", fontproperties=font_prop)
    ax.legend(prop=font_prop)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    fig_all.append("PR")

    # ----- 5) 分数 vs 标签 -----
    n_show = min(2000, len(scores))
    t = np.arange(n_show)

    fig4, ax = plt.subplots(figsize=(12, 3))
    ax.fill_between(t, 0, 1, where=y_true[:n_show] > 0.5, alpha=0.25, color="red", label="真实异常")
    ax.plot(t, scores[:n_show], lw=0.5, color="C0", label="异常分数")
    ax.set_xlabel("时间步", fontproperties=font_prop)
    ax.set_ylabel("异常分数", fontproperties=font_prop)
    ax.set_title(f"异常分数 vs 真实标签（前 {n_show} 步）", fontproperties=font_prop)
    ax.legend(prop=font_prop)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    fig_all.append("时序对比")

    # ----- 6) 预测 vs 真实 -----
    fig5, axes = plt.subplots(2, 1, figsize=(12, 3.5), sharex=True)

    axes[0].plot(t, y_true[:n_show], drawstyle="steps-post", color="C1", label="真实标签")
    axes[0].set_ylabel("真实", fontproperties=font_prop)
    axes[0].legend(prop=font_prop)
    axes[0].set_ylim(-0.1, 1.2)

    axes[1].plot(t, y_pred[:n_show], drawstyle="steps-post", color="C2", label="预测标签")
    axes[1].set_ylabel("预测", fontproperties=font_prop)
    axes[1].set_xlabel("时间步", fontproperties=font_prop)
    axes[1].legend(prop=font_prop)
    axes[1].set_ylim(-0.1, 1.2)

    axes[0].set_title(f"真实 vs 预测（前 {n_show} 步）", fontproperties=font_prop)

    plt.tight_layout()
    plt.show()
    fig_all.append("预测对比")

    print("已绘制:", ", ".join(fig_all))

except NameError as e:
    print("缺少 lossFinal / labelsFinal / lossTfinal / result：请先运行训练。", e)

## 6.2 对比学习实验模板（TranAD + CL）

在已训练的 TranAD 基础上，加入 **InfoNCE 对比损失** 做联合微调（重构 + 对比），验证联合训练方案。

- 输入：第 5 节已产生的 `model`、`trainD`、`testD`、`labels`、`lossTfinal/lossFinal/labelsFinal/result`。
- 输出：CL 微调后的指标、与 baseline 的对比表、以及 F1/Precision/Recall 的柱状图。
- 说明：这是一个可直接运行的实验模板（便于后续改增强策略/损失权重/温度等超参）。

In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import pandas as pd


# =========================
# 0) 基本检查
# =========================
required_vars = ["model", "trainD", "testD", "labels", "pot_eval", "result"]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(f"缺少变量: {missing}。请先完整运行第 5 节训练与测试。")

# 保存 baseline 指标（第 5 节结果）
baseline_result = dict(result)

# =========================
# 1) CL 超参数（可改）
# =========================
CL_EPOCHS = 5
CL_BATCH = 128
CL_LR = 5e-4
CL_WEIGHT_DECAY = 1e-5
LAMBDA_CL = 0.005
TEMPERATURE = 0.2
CLIP_NORM = 1.0

# =========================
# 2) 增强函数（可替换为你的策略）
# =========================
def jitter(x, sigma=0.01):
    return x + torch.randn_like(x) * sigma

def scaling(x, sigma=0.1):
    # x: [B, W, F]
    B, _, Fdim = x.shape
    factor = torch.normal(mean=torch.ones(B, 1, Fdim, device=x.device, dtype=x.dtype),
                          std=sigma)
    return x * factor

def time_mask(x, mask_ratio=0.1):
    if mask_ratio <= 0:
        return x
    B, W, _ = x.shape
    m = max(1, int(W * mask_ratio))
    x2 = x.clone()
    for b in range(B):
        st = int(torch.randint(0, max(1, W - m + 1), (1,), device=x.device))
        x2[b, st:st + m, :] = 0.0
    return x2

def feature_mask(x, mask_ratio=0.05):
    if mask_ratio <= 0:
        return x
    B, _, Fdim = x.shape
    m = max(1, int(Fdim * mask_ratio))
    x2 = x.clone()
    for b in range(B):
        idx = torch.randperm(Fdim, device=x.device)[:m]
        x2[b, :, idx] = 0.0
    return x2

def augment_ts(x):
    x1 = jitter(x, sigma=0.01)
    x1 = scaling(x1, sigma=0.08)
    x1 = time_mask(x1, mask_ratio=0.1)
    x1 = feature_mask(x1, mask_ratio=0.05)
    return x1

# =========================
# 3) InfoNCE 损失
# =========================
def info_nce_loss(z1, z2, temperature=0.2):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    logits = (z1 @ z2.t()) / temperature
    labels_ce = torch.arange(z1.size(0), device=z1.device, dtype=torch.long)
    loss1 = F.cross_entropy(logits, labels_ce)
    loss2 = F.cross_entropy(logits.t(), labels_ce)
    return 0.5 * (loss1 + loss2)

# =========================
# 4) 在 TranAD 上接投影头
# =========================
# 使用一个 batch 动态推断 representation 维度
_device = next(model.parameters()).device
_dtype = next(model.parameters()).dtype

with torch.no_grad():
    x0 = trainD[:min(CL_BATCH, len(trainD))].to(device=_device, dtype=_dtype)
    src0 = x0.permute(1, 0, 2)
    tgt0 = src0[-1:, :, :]
    c0 = torch.zeros_like(src0)
    _, mem0 = model.encode(src0, c0, tgt0)  # mem: [W, B, 2F]
    rep_dim = mem0.mean(dim=0).shape[1]

proj_head = nn.Sequential(
    nn.Linear(rep_dim, rep_dim),
    nn.ReLU(inplace=True),
    nn.Linear(rep_dim, 128),
).to(device=_device, dtype=_dtype)

optimizer_cl = torch.optim.AdamW(
    list(model.parameters()) + list(proj_head.parameters()),
    lr=CL_LR,
    weight_decay=CL_WEIGHT_DECAY,
)
recon_criterion = nn.MSELoss(reduction="none")

# DataLoader（trainD 是第 5 节转换后的窗口）
train_dataset_cl = TensorDataset(trainD, trainD)
train_loader_cl = DataLoader(train_dataset_cl, batch_size=CL_BATCH, shuffle=True)

# 可选：保存初始模型用于回退
model_before_cl = copy.deepcopy(model.state_dict())

# =========================
# 5) CL 联合微调（重构 + 对比）
# =========================
hist = []
model.train()
for ep in range(1, CL_EPOCHS + 1):
    ep_recon, ep_cl, n_samp = 0.0, 0.0, 0

    for xb, _ in train_loader_cl:
        xb = xb.to(device=_device, dtype=_dtype)
        B = xb.size(0)

        # 重构分支（TranAD 原逻辑）
        src = xb.permute(1, 0, 2)
        elem = src[-1:, :, :]
        z = model(src, elem)
        if isinstance(z, tuple):
            z_use = z[1]
        else:
            z_use = z
        loss_recon = torch.mean(recon_criterion(z_use, elem))

        # 对比分支
        xb1 = augment_ts(xb)
        xb2 = augment_ts(xb)
        src1 = xb1.permute(1, 0, 2)
        src2 = xb2.permute(1, 0, 2)
        tgt1 = src1[-1:, :, :]
        tgt2 = src2[-1:, :, :]
        c1 = torch.zeros_like(src1)
        c2 = torch.zeros_like(src2)
        _, mem1 = model.encode(src1, c1, tgt1)
        _, mem2 = model.encode(src2, c2, tgt2)
        rep1 = mem1.mean(dim=0)
        rep2 = mem2.mean(dim=0)
        z1 = proj_head(rep1)
        z2 = proj_head(rep2)
        loss_cl = info_nce_loss(z1, z2, temperature=TEMPERATURE)

        loss = loss_recon + LAMBDA_CL * loss_cl

        optimizer_cl.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(model.parameters()) + list(proj_head.parameters()), CLIP_NORM)
        optimizer_cl.step()

        ep_recon += float(loss_recon.item()) * B
        ep_cl += float(loss_cl.item()) * B
        n_samp += B

    ep_recon /= max(n_samp, 1)
    ep_cl /= max(n_samp, 1)
    ep_total = ep_recon + LAMBDA_CL * ep_cl
    hist.append((ep_recon, ep_cl, ep_total))
    print(f"[CL] epoch={ep:02d} recon={ep_recon:.6f} cl={ep_cl:.6f} total={ep_total:.6f}")

# =========================
# 6) CL 后评估（沿用 TranAD 原评估协议）
# =========================
model.eval()
with torch.no_grad():
    loss_cl_test, _ = backprop(0, model, testD, testO, optimizer_cl, optimizer_cl, training=False)
    loss_cl_train, _ = backprop(0, model, trainD, trainO, optimizer_cl, optimizer_cl, training=False)

lossTfinal_cl = np.mean(loss_cl_train, axis=1)
lossFinal_cl = np.mean(loss_cl_test, axis=1)
labelsFinal_cl = (np.sum(labels, axis=1) >= 1) + 0
result_cl, pred_cl = pot_eval(lossTfinal_cl, lossFinal_cl, labelsFinal_cl)

# 兼容原 notebook 里的额外指标
try:
    result_cl.update(hit_att(loss_cl_test, labels))
    result_cl.update(ndcg(loss_cl_test, labels))
except Exception:
    pass

print("\n--- TranAD + CL POT 评估指标 ---")
for k, v in result_cl.items():
    if isinstance(v, (float, np.floating)):
        print(f"  {k}: {float(v):.4f}")
    else:
        print(f"  {k}: {v}")

# =========================
# 7) baseline vs CL 对比表 + 图
# =========================
keys_show = ["f1", "precision", "recall", "ROC/AUC", "threshold", "TP", "TN", "FP", "FN"]
rows = []
for k in keys_show:
    rows.append({
        "metric": k,
        "baseline_tranad": baseline_result.get(k, np.nan),
        "tranad_plus_cl": result_cl.get(k, np.nan),
    })
cmp_df = pd.DataFrame(rows)
display(cmp_df)

plot_keys = ["f1", "precision", "recall"]
b = [float(baseline_result.get(k, np.nan)) for k in plot_keys]
c = [float(result_cl.get(k, np.nan)) for k in plot_keys]
x = np.arange(len(plot_keys))
w = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w/2, b, width=w, label="TranAD baseline")
ax.bar(x + w/2, c, width=w, label="TranAD + CL")
ax.set_xticks(x)
ax.set_xticklabels(plot_keys)
ax.set_ylim(0, 1.05)
ax.set_title("Baseline vs CL (POT metrics)")
ax.grid(True, axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

# 可直接复用到后续可视化
cl_artifacts = {
    "lossTfinal_cl": lossTfinal_cl,
    "lossFinal_cl": lossFinal_cl,
    "labelsFinal_cl": labelsFinal_cl,
    "pred_cl": pred_cl,
    "result_cl": result_cl,
    "train_hist_cl": hist,
}
print("\n已生成 cl_artifacts，可用于后续画图与导出。")


In [ ]:
# =========================
# 评估可视化（自动选择 CL 结果或 baseline）
# 输出：混淆矩阵、分类报告、ROC/AUC
# =========================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

# 1) 自动选择可用变量（优先 CL）
if "lossFinal_cl" in globals() and "labelsFinal_cl" in globals():
    scores = np.asarray(lossFinal_cl).ravel().astype(float)
    y_true = np.asarray(labelsFinal_cl).ravel().astype(int)
    if "pred_cl" in globals():
        y_pred = np.asarray(pred_cl).ravel().astype(int)
    elif "pot_eval" in globals() and "lossTfinal_cl" in globals():
        _, y_pred = pot_eval(lossTfinal_cl, lossFinal_cl, labelsFinal_cl)
        y_pred = np.asarray(y_pred).ravel().astype(int)
    else:
        # 兜底：按 99 分位阈值二值化
        th = np.percentile(scores, 99)
        y_pred = (scores > th).astype(int)
    run_tag = "CL"
elif "lossFinal" in globals() and "labelsFinal" in globals():
    scores = np.asarray(lossFinal).ravel().astype(float)
    y_true = np.asarray(labelsFinal).ravel().astype(int)
    if "pot_eval" in globals() and "lossTfinal" in globals():
        _, y_pred = pot_eval(lossTfinal, lossFinal, labelsFinal)
        y_pred = np.asarray(y_pred).ravel().astype(int)
    else:
        th = np.percentile(scores, 99)
        y_pred = (scores > th).astype(int)
    run_tag = "Baseline"
else:
    raise RuntimeError("缺少可用变量：请先运行训练/评估单元，至少生成 lossFinal(或 lossFinal_cl) 与 labelsFinal(或 labelsFinal_cl)。")

# 保证长度一致
n = min(len(scores), len(y_true), len(y_pred))
scores, y_true, y_pred = scores[:n], y_true[:n], y_pred[:n]

print(f"当前评估对象: {run_tag} | 样本数: {n}")

# 2) 标签方向统一（以分类报告为准）
# 约定：0=normal(negative), 1=anomaly(positive)
y_true = (np.asarray(y_true).ravel() > 0).astype(int)
y_pred = (np.asarray(y_pred).ravel() > 0).astype(int)
report_labels = [0, 1]
report_names = ["normal", "anomaly"]

print("\n=== Classification Report (0=normal, 1=anomaly) ===")
print(
    classification_report(
        y_true,
        y_pred,
        labels=report_labels,
        target_names=report_names,
        digits=4,
        zero_division=0,
    )
)

# 3) 混淆矩阵（labels 顺序与分类报告完全一致）
cm = confusion_matrix(y_true, y_pred, labels=report_labels)
fig, ax = plt.subplots(1, 1, figsize=(5.5, 4.8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=report_names,
    yticklabels=report_names,
    ax=ax,
)
ax.set_title(f"Confusion Matrix ({run_tag})")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.tight_layout()
plt.show()

# 4) ROC / AUC
# 默认 scores 越大越异常，且正类固定为 anomaly(1)
fpr, tpr, _ = roc_curve(y_true, scores, pos_label=1)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(1, 1, figsize=(5.8, 4.8))
ax.plot(fpr, tpr, lw=2, label=f"ROC (AUC = {roc_auc:.4f})")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"ROC Curve ({run_tag})")
ax.legend(loc="lower right")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print(f"AUC = {roc_auc:.6f}")